# NB06: Revenue Feature Engineering

Builds revenue-specific features on top of NB03's modeling dataset.
NB03 is designed for unit forecasting — this notebook adds features
that capture revenue dynamics: price mix, longer revenue memory,
high-value transaction patterns, and store-level revenue signals.

**Input:** `data/processed/modeling_dataset.csv` (from NB03)
**Output:** `data/processed/revenue_modeling_dataset.csv` (consumed by NB07)

**Features added:**
1. Longer revenue lags (8w, 12w) and rolling stats — captures off-season patterns
2. Revenue volatility (CV) — lets model segment high-variance groups
3. Price percentiles per division — normalizes price across product mix
4. High-value transaction flag — identifies weeks with outlier sales
5. Store-level weekly revenue — proxy for foot traffic / store activity
6. Intermittency features — cumulative sales count, zero fraction
7. Feature manifest — saved alongside dataset for NB07 to read dynamically

---
## Cell 1: Setup
---

In [8]:
import sys, os

_this_dir = os.path.dirname(os.path.abspath('__file__'))
_candidates = [
    os.environ.get('UC4_PROJECT_ROOT', ''),
    os.path.join(_this_dir, '..'),
]
for _c in _candidates:
    _test = os.path.join(_c, 'data', 'processed', 'modeling_dataset.csv')
    if os.path.exists(_test):
        _project_root = _c
        break
else:
    raise FileNotFoundError("Cannot find project root. Set UC4_PROJECT_ROOT or run from notebooks/")

_pylibs = os.path.join(os.path.dirname(_project_root), '.pylibs')
if os.path.isdir(_pylibs):
    sys.path.insert(0, _pylibs)

import pandas as pd
import numpy as np
from pathlib import Path
import json as _json
import warnings
warnings.filterwarnings('ignore')

project_root = Path(_project_root)
data_dir = project_root / 'data' / 'processed'

print("=" * 60)
print("NB06: REVENUE FEATURE ENGINEERING")
print("=" * 60)

NB06: REVENUE FEATURE ENGINEERING


---
## Cell 2: Load NB03 Output
---

In [9]:
df = pd.read_csv(data_dir / 'modeling_dataset.csv')
df['week_ending'] = pd.to_datetime(df['week_ending'])
df = df.sort_values(['store_code', 'division_code', 'week_ending']).reset_index(drop=True)

GROUP_COLS = ['store_code', 'division_code']

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Stores: {df['store_code'].nunique()}, Divisions: {df['division_code'].nunique()}")
print(f"Groups: {df.groupby(GROUP_COLS).ngroups}")
print(f"Date range: {df['week_ending'].min().date()} to {df['week_ending'].max().date()}")

# Track which columns came from NB03 vs added here
nb03_cols = set(df.columns)
print(f"\nNB03 columns: {len(nb03_cols)}")

Loaded: 24,323 rows x 81 cols
Stores: 27, Divisions: 11
Groups: 287
Date range: 2023-11-05 to 2026-03-22

NB03 columns: 81


---
## Cell 3: Revenue-Specific Lag & Rolling Features

NB03 provides `revenue_lag_1w`, `revenue_lag_4w`, `revenue_roll4_mean`.
We add longer memory (8w, 12w) which helps in off-season when sales are sparse,
plus revenue volatility and rolling stats on avg_price_per_unit.
---

In [10]:
def add_revenue_features(group):
    group = group.sort_values('week_ending').copy()
    rev = group['revenue']
    rev_shifted = rev.shift(1)

    # ── Longer revenue lags ──
    group['revenue_lag_2w']  = rev.shift(2)
    group['revenue_lag_8w']  = rev.shift(8)
    group['revenue_lag_12w'] = rev.shift(12)
    group['revenue_lag_52w'] = rev.shift(52)

    # ── Longer rolling windows ──
    group['revenue_roll8_mean']  = rev_shifted.rolling(8, min_periods=4).mean()
    group['revenue_roll8_std']   = rev_shifted.rolling(8, min_periods=4).std()
    group['revenue_roll12_mean'] = rev_shifted.rolling(12, min_periods=6).mean()
    group['revenue_roll12_std']  = rev_shifted.rolling(12, min_periods=6).std()

    # ── Revenue volatility (CV) ──
    roll4_mean = group.get('revenue_roll4_mean', rev_shifted.rolling(4, min_periods=2).mean())
    roll4_std = rev_shifted.rolling(4, min_periods=2).std()
    group['revenue_volatility'] = roll4_std / roll4_mean.replace(0, np.nan)

    # ── Price rolling stats (captures product mix shifts) ──
    if 'avg_price_per_unit' in group.columns:
        price = group['avg_price_per_unit'].shift(1)
        group['price_roll4_mean'] = price.rolling(4, min_periods=2).mean()
        group['price_roll4_std']  = price.rolling(4, min_periods=2).std()
        group['price_momentum']   = group['avg_price_per_unit'] - price

    return group

# Apply per group (loop+concat to avoid pandas groupby.apply column loss)
_parts = []
for _keys, _grp in df.groupby(GROUP_COLS):
    _parts.append(add_revenue_features(_grp))
df = pd.concat(_parts, ignore_index=True)

new_rev_feats = [c for c in df.columns if c not in nb03_cols]
print(f"Added {len(new_rev_feats)} revenue features: {new_rev_feats}")


Added 12 revenue features: ['revenue_lag_2w', 'revenue_lag_8w', 'revenue_lag_12w', 'revenue_lag_52w', 'revenue_roll8_mean', 'revenue_roll8_std', 'revenue_roll12_mean', 'revenue_roll12_std', 'revenue_volatility', 'price_roll4_mean', 'price_roll4_std', 'price_momentum']


---
## Cell 4: Price Percentiles & High-Value Flags

Computes price percentiles per division (from training data),
then flags weeks where avg_price_per_unit exceeds the division's P75.
This helps the model recognize when a high-value item sale happened.
---

In [11]:
if 'avg_price_per_unit' in df.columns:
    # Division-level price percentiles
    price_stats = (
        df[df['avg_price_per_unit'].notna() & (df['avg_price_per_unit'] > 0)]
        .groupby('division_code')['avg_price_per_unit']
        .describe(percentiles=[0.25, 0.5, 0.75])
        .rename(columns={'25%': 'price_p25', '50%': 'price_p50', '75%': 'price_p75'})
        [['price_p25', 'price_p50', 'price_p75']]
    )

    df = df.merge(price_stats, on='division_code', how='left')

    # Price relative to division median — raw, no clipping (LightGBM handles outliers natively)
    df['price_vs_div_median'] = df['avg_price_per_unit'] / df['price_p50'].replace(0, np.nan)

    # High-value flag: avg_price above P75 for the division
    df['high_value_week'] = (df['avg_price_per_unit'] > df['price_p75']).astype(int)

    # Drop raw percentile columns (static per division, captured by div_enc)
    df = df.drop(columns=['price_p25', 'price_p50', 'price_p75'])

    print("Added: price_vs_div_median, high_value_week")
else:
    print("Skipped price features (avg_price_per_unit not found)")


Added: price_vs_div_median, high_value_week


---
## Cell 5: Store-Level Revenue (Cross-Division Signal)

Total store revenue across all divisions in the same week.
Proxy for store foot traffic — if total store revenue is high,
individual divisions are more likely to sell too.
---

In [12]:
store_weekly_rev = (
    df.groupby(['store_code', 'week_ending'])['revenue']
    .sum()
    .reset_index()
    .rename(columns={'revenue': 'store_weekly_revenue'})
)

df = df.merge(store_weekly_rev, on=['store_code', 'week_ending'], how='left')

# Lagged version (avoid leakage — use last week's store revenue)
df = df.sort_values(['store_code', 'division_code', 'week_ending'])
df['store_revenue_lag_1w'] = df.groupby(GROUP_COLS)['store_weekly_revenue'].shift(1)

print(f"Added: store_weekly_revenue, store_revenue_lag_1w")

Added: store_weekly_revenue, store_revenue_lag_1w


---
## Cell 6: Intermittency Features

Moved from NB07 — these belong in feature engineering, not in the model notebook.
---

In [13]:
df = df.sort_values(['store_code', 'division_code', 'week_ending'])

# Cumulative sales count (how many weeks with sales so far)
nonzero = (df['units'] > 0).astype(int)
df['cumulative_sales_count'] = nonzero.groupby(
    [df['store_code'], df['division_code']]
).cumsum()

# Zero fraction in last 12 weeks
df['zero_frac_12w'] = 1 - df.groupby(GROUP_COLS)['units'].transform(
    lambda x: x.rolling(12, min_periods=4).apply(lambda w: (w > 0).mean())
)

# Revenue-specific: zero revenue fraction
df['zero_rev_frac_12w'] = 1 - df.groupby(GROUP_COLS)['revenue'].transform(
    lambda x: x.rolling(12, min_periods=4).apply(lambda w: (w > 0).mean())
)

print("Added: cumulative_sales_count, zero_frac_12w, zero_rev_frac_12w")

Added: cumulative_sales_count, zero_frac_12w, zero_rev_frac_12w


---
## Cell 7: Save Output + Feature Manifest
---

In [14]:
# ── Save dataset ──
df.to_csv(data_dir / 'revenue_modeling_dataset.csv', index=False)
print(f"Saved: revenue_modeling_dataset.csv ({df.shape[0]:,} rows x {df.shape[1]} cols)")

# ── Feature manifest: tells NB07 which features are available ──
all_new_cols = [c for c in df.columns if c not in nb03_cols]
manifest = {
    'source': 'NB06',
    'base_dataset': 'modeling_dataset.csv',
    'output_dataset': 'revenue_modeling_dataset.csv',
    'nb03_columns': sorted(nb03_cols),
    'nb06_added_columns': sorted(all_new_cols),
    'all_numeric_features': sorted([
        c for c in df.select_dtypes(include=[np.number]).columns
        if c not in ['units', 'revenue', 'store_enc', 'div_enc']
    ]),
    'revenue_specific_features': sorted([
        c for c in all_new_cols
        if c not in ['store_enc', 'div_enc']
    ]),
}

with open(data_dir / 'nb06_feature_manifest.json', 'w') as f:
    _json.dump(manifest, f, indent=2)

print(f"Saved: nb06_feature_manifest.json")
print(f"\nNB03 columns: {len(nb03_cols)}")
print(f"NB06 added:   {len(all_new_cols)}")
print(f"Total:        {df.shape[1]}")
print(f"\nNew features ({len(all_new_cols)}):")
for c in sorted(all_new_cols):
    print(f"  - {c}")

Saved: revenue_modeling_dataset.csv (24,323 rows x 100 cols)
Saved: nb06_feature_manifest.json

NB03 columns: 81
NB06 added:   19
Total:        100

New features (19):
  - cumulative_sales_count
  - high_value_week
  - price_momentum
  - price_roll4_mean
  - price_roll4_std
  - price_vs_div_median
  - revenue_lag_12w
  - revenue_lag_2w
  - revenue_lag_52w
  - revenue_lag_8w
  - revenue_roll12_mean
  - revenue_roll12_std
  - revenue_roll8_mean
  - revenue_roll8_std
  - revenue_volatility
  - store_revenue_lag_1w
  - store_weekly_revenue
  - zero_frac_12w
  - zero_rev_frac_12w
